In [59]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


In [60]:
train = np.load("train.npz")
test = np.load("test.npz")

In [61]:
train_data = train["data"]
train_labels = train["labels"]
test_data = test["data"]

In [62]:
print("Full train data:", train_data.shape)
print("Full train labels:", train_labels.shape)
print("Test data:", test_data.shape)

Full train data: (60000, 784)
Full train labels: (60000,)
Test data: (15000, 784)


In [63]:
# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_data,
    train_labels,
    test_size=0.2,
    random_state=1,
    stratify=train_labels,
)

In [64]:
print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)

Training split: (48000, 784)
Validation split: (12000, 784)


In [65]:
def one_hot(y, num_classes=10):
    mat = np.eye(num_classes)[np.array(y).reshape(-1)]
    return mat

y = [0, 2, 1]
nb_classes = 3
print(one_hot(y, nb_classes))

[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


In [66]:
def softmax(z):
    z = z - np.max(z, axis=-1, keepdims=True)
    
    expsum = np.sum(np.exp(z), axis=-1, keepdims=True)
    return np.exp(z) / expsum

z = np.array([1.0, 2.0, 3.0])
print(softmax(z), np.sum(softmax(z)))

z = np.array([1000.0, 1001.0, 1002.0])
print(softmax(z), np.sum(softmax(z))) # hits overflow, needs z - max(z)

z = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])
print(softmax(z), np.sum(softmax(z), axis=-1, keepdims=True)) # hits size mismatch, needs keepdims

[0.09003057 0.24472847 0.66524096] 0.9999999999999999
[0.09003057 0.24472847 0.66524096] 0.9999999999999999
[[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]] [[1.]
 [1.]]


In [67]:
def forward(X, W, b):
    # prediction
    z = X @ W + b

    # activation
    y_hat = softmax(z)

    return y_hat

# X.shape = (48000, 784)
# W.shape = (784, 10)
# b.shape = (10, )
X = np.random.randn(48000, 784)
W = np.random.randn(784, 10) * np.sqrt(2 / 784) # He initialization
b = np.zeros(10)
print(forward(X, W, b)[:2])

[[0.29587846 0.00514953 0.03195197 0.12354542 0.01830881 0.03143391
  0.38145306 0.00921176 0.05651061 0.04655648]
 [0.02416564 0.01942987 0.13187661 0.35285523 0.00648836 0.22129461
  0.17591966 0.0191044  0.01532797 0.03353766]]


In [68]:
def cross_entropy(y_hat, y, eps=1e-15):
    m = y_hat.shape[0]
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -1/m * np.sum(y * np.log(y_hat))

y = one_hot([0, 2], num_classes=3)
y_hat = np.array([
    [0.7, 0.2, 0.1],  
    [0.1, 0.1, 0.8]   
])
print(cross_entropy(y_hat, y))

y_hat = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0]
])
print(cross_entropy(y_hat, y)) # hits division by log 0, needs clipping

0.2899092476264711
9.992007221626415e-16


In [69]:
def backward(X, y_hat, y, W, b, lr):
    m = y_hat.shape[0]

    # dout = dL/dy_hat * dy_hat/dz
    dout = (y_hat - y) / m

    # dW = dL/dy_hat * dy_hat/dz * dz/dW = dout * X
    # X.shape = (48000, 748)
    # dout.shape = (48000, 10)
    dW = X.T @ dout

    # sum of dout across samples
    db = np.sum(dout, axis=0)

    # gradient descent
    W -= lr * dW
    b -= lr * db

    return W, b

X = np.array([[1.0, 2.0], [3.0, 4.0]])
y = one_hot([0, 1], num_classes=2)
W = np.zeros((2, 2))
b = np.zeros(2)
y_hat = np.array([[0.8, 0.2], [0.1, 0.9]])

W_new, b_new = backward(X, y_hat, y, W, b, lr=0.1)
print(W_new, b_new)

[[-5.00000000e-03  5.00000000e-03]
 [-5.55111512e-18 -5.55111512e-18]] [ 0.005 -0.005]


In [70]:
def predict(X, W, b):
    y_hat = forward(X, W, b)
    winner = np.argmax(y_hat, axis=-1)

    return winner

X = np.array([[1.0, 0.0], [0.0, 1.0]])
W = np.array([[10.0, -10.0], [-10.0, 10.0]])
b = np.zeros(2)
print(predict(X, W, b))

print(predict(X[0], W, b)) 

[0 1]
0


In [84]:
# setup
# X_train_norm = X_train / 255.0 if X_train.max() > 1.0 else X_train
# X_val_norm = X_val / 255.0 if X_val.max() > 1.0 else X_val
X_norm = train_data / 255.0 if train_data.max() > 1.0 else train_data
X_test_norm = test_data / 255.0 if test_data.max() > 1.0 else test_data

# y_train_oh = one_hot(y_train, num_classes=10)
# y_val_oh = one_hot(y_val, num_classes=10)
y_oh = one_hot(train_labels, num_classes=10)

np.random.seed(42)
W = np.random.randn(784, 10) * np.sqrt(2 / 784) # He init
b = np.zeros(10)

# hyperparameters
epochs = 50
batch_size = 64
lr = 0.1
m = X_norm.shape[0]

In [85]:
# mini batch gsd
for epoch in range(epochs):
    indices = np.random.permutation(m)
    X_shuf = X_norm[indices]
    y_shuf = y_oh[indices]

    # loop over batches
    for i in range(0, m, batch_size):
        X_batch = X_shuf[i:i+batch_size]
        y_batch = y_shuf[i:i+batch_size]

        y_hat = forward(X_batch, W, b)
        W, b = backward(X_batch, y_hat, y_batch, W, b, lr)

    # log progrees
    if (epoch + 1) % 5 == 0 or epoch == 0:
        val_probs = forward(X_val_norm, W, b)
        val_loss = cross_entropy(val_probs, y_val_oh)
        val_acc = np.mean(predict(X_val_norm, W, b) == y_val)
        print(f"Epoch {epoch + 1}/{epochs} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")     

    lr *= 0.95   

Epoch 1/50 | Val Loss: 0.3600 | Val Acc: 0.9012
Epoch 5/50 | Val Loss: 0.2969 | Val Acc: 0.9181
Epoch 10/50 | Val Loss: 0.2804 | Val Acc: 0.9220
Epoch 15/50 | Val Loss: 0.2746 | Val Acc: 0.9239
Epoch 20/50 | Val Loss: 0.2704 | Val Acc: 0.9258
Epoch 25/50 | Val Loss: 0.2677 | Val Acc: 0.9261
Epoch 30/50 | Val Loss: 0.2663 | Val Acc: 0.9258
Epoch 35/50 | Val Loss: 0.2657 | Val Acc: 0.9273
Epoch 40/50 | Val Loss: 0.2646 | Val Acc: 0.9280
Epoch 45/50 | Val Loss: 0.2638 | Val Acc: 0.9274
Epoch 50/50 | Val Loss: 0.2634 | Val Acc: 0.9278


In [86]:
# final eval
train_accuracy = np.mean(predict(X_train_norm, W, b) == y_train)
val_accuracy = np.mean(predict(X_val_norm, W, b) == y_val)
predictions = predict(X_test_norm, W, b)

In [87]:
print("Training accuracy:", train_accuracy)
print("Validation accuracy:", val_accuracy)

Training accuracy: 0.9293125
Validation accuracy: 0.9278333333333333


In [88]:
submission = pd.DataFrame({
    "ID": range(len(predictions)),
    "target": predictions,
})

In [89]:
submission.to_csv("submission.csv", index=False)
print("Created submission.csv")

Created submission.csv
